# Wykrywanie obiektów obcych na pasie startowym (YOLO)
Szkielet projektu. Pełne wyjaśnienia krok po kroku: `docs/dziennik.md`.

**Dwa poziomy**
1. **Anomalia (pierwszy próg sukcesu):** czy na pasie jest obcy element, i gdzie. Wynik to alert, żeby ekipa mogła go usunąć przed kolejnym startem.
2. **Rozpoznanie (później):** co to jest (śrubka, puszka, butelka), albo „do rozpoznania".

Ten notebook obejmuje na razie poziom 1 w wersji szkieletowej (funkcje to `TODO`).

## 1. Cel i zakres
YOLO, jedna klasa „obiekt obcy". Podstawowa jednostka to pojedynczy obraz (zdjęcie lub klatka z nagrania), bez analizy w czasie rzeczywistym. Położenie = identyfikator zdjęcia i ramka w pikselach (bez metrów i bez pozycji na pasie, bo brakuje metadanych).

## 2. Konfiguracja
Wszystkie parametry w `config.yaml`. Dzięki temu zmieniamy eksperyment bez ruszania kodu.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.config import load_config

cfg = load_config()
cfg["detect"]

## 3. Dane
Zdjęcia nie mają jeszcze ramek. Podział train/val/test **po obiekcie (FOD_xx)**, żeby to samo zdjęcie ujęte dwa razy nie trafiło do różnych zbiorów (przeciek danych).

In [ ]:
from src.data import parse_filename

parse_filename("FOD_01_GT_20260824_114626.jpg")

## 4. Model
Mały YOLO wstępnie wytrenowany (transfer learning), douczany na jednej klasie. Dlaczego tak: mało zdjęć, więc uczymy się od modelu, który już „widział świat".

In [ ]:
from src.detect import load_model

try:
    model = load_model(cfg["detect"]["model"])
except NotImplementedError as e:
    print("Szkielet:", e)

## 5. Detekcja i zapis wyników
CSV + obrazy z ramkami. Każdy wynik to ramka i pewność. Alert (`src/report.py`) niesie status: wykryty / rozpoznany / do rozpoznania.

In [ ]:
from src.report import FlightReport

report = FlightReport()
print("Pas czysty (szkielet, nic nie przeanalizowano):", report.runway_clear)

## 6. Ewaluacja
Poziom obrazu: TP, FP, TN, FN, precision, recall, F1. Lokalizacja: IoU. Priorytet: wysoki recall, bo przeoczony obiekt jest groźniejszy niż fałszywy alarm.

In [ ]:
from src.evaluate import iou

iou((0, 0, 10, 10), (5, 5, 15, 15))

## 7. Eksperyment
Wpływ progu pewności na liczbę poprawnych i błędnych wykryć. Wspólny protokół (stały podział testowy, te same metryki), żeby porównać YOLO z innymi metodami.

In [ ]:
cfg["experiment"]["conf_sweep"]

## 8. Wyniki i wizualizacje
_Brak, szkielet. Tu pojawią się tabele i wykresy._

## 9. Wnioski i ograniczenia
_Do uzupełnienia. Pytanie otwarte: czy YOLO będzie na końcu optymalny? Rekomendacja ma wynikać z wyników na wspólnym teście, nie z założenia._